## Import Libraries & Load Data

We start by importing the **necessary libraries** and **reading the CSV file** into a DataFrame:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub as kh
from scipy import stats
from scipy.stats import shapiro, anderson, normaltest, skew, kurtosis
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, confusion_matrix, classification_report
    )

path = kh.dataset_download("solomonameh/spotify-music-dataset")

df_high = pd.read_csv(path + "/high_popularity_spotify_data.csv")
df_low = pd.read_csv(path + "/low_popularity_spotify_data.csv")

pd.options.mode.chained_assignment = None

In [ ]:
df_low['popularity'] = 'Low'
df_high['popularity'] = 'High'

In [ ]:
print("df_high column names:")
for i, col in enumerate(df_high.columns, 1):
    print(f"{i:2d}.'{col}'")




In [ ]:
print("df_low column names:")
for i, col in enumerate(df_low.columns, 1):
    print(f"{i:2d}.'{col}'")




In [ ]:
df = pd.concat([df_low,df_high], ignore_index=True)
df

## Data Cleaning

**Firstly**, we clean the dataset by performing the following checks:

- ✅ Check for **NaN** (missing) values  
- ✅ Check for **duplicate rows**  
- ✅ Check for **non-NaN values** 


In [ ]:
#check for duplicate rows
duplicates = df.duplicated()
number_duplicates = duplicates.sum()
print(f"Number of duplicate rows: {number_duplicates}")

#check for Nan values
Nan_values = df.isna().sum().sum()
print(f"Number of Nan values: {Nan_values}")

#check for non-Nan values
Non_Nan = df.isin(['?', '--', 'N/A', 'na', '', 'null', '0']).any().sum()
print(f"Number of non Nan values: {Non_Nan}")


In [ ]:
df[df.isna().any(axis=1)]


## Data Inspection

After checking the dataset, it appears to be **pretty clean**:

- There are few **NaN (missing) values**  
- There are no **duplicate rows**  

✅ **Conclusion:** since there are only 2 null rows, we can drop them in order to have only complete rows.

In [ ]:
df = df.dropna()
df.isna().sum().sum()

## Dropping Unnecessary Columns

By inspecting the dataset, we notice that some columns are **not needed** for our further analysis.  

✅ **Conclusion:** Drop the unnecessary columns from the DataFrame.


In [ ]:
#remove unnecesarry columns
df.drop(columns=['track_album_id', 'id','playlist_id'], inplace=True)

In [ ]:
#remove unnecesarry columns
df.drop(columns=['track_href', 'uri','analysis_url'], inplace=True)

In [ ]:
#remove unnecesarry columns
df.drop(columns=['track_id'], inplace=True)

In [ ]:
#the column 'type' seems to have the same rows
df['type']
column = 'type'
all_same = df['type'].nunique()==1
all_same
df.drop(columns=['type'], inplace=True) 

## Summary Statistics for Numerical Features

We calculate **summary statistics** for the numerical features in the dataset to:

- Understand the **central tendency** (mean, median)  
- Assess the **spread** (standard deviation, min, max)  
- Detect any **outliers** or unusual values  
- Gain insights for **data transformation and modeling**


In [ ]:
#Summary statistics
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
print('='*80)
print('Numerical features Summary Statistics')
print('='*80)
df[numerical_cols].describe().T

## Skewness and Kurtosis Analysis

We compute the **skewness** and **kurtosis** of the numerical features to:

- Measure the **asymmetry** of each feature's distribution (**skewness**)  
- Measure the **tailedness** or outlier-proneness (**kurtosis**)  
- Identify features that may need **transformation** before modeling  


In [ ]:
#Skewness and kurtosis calculation
skewness_values = df[numerical_cols].apply(lambda x: skew(x.dropna()) if x.notna().sum() > 1 else np.nan)
kurtosis_values = df[numerical_cols].apply(lambda x: kurtosis(x.dropna()) if x.notna().sum() > 1 else np.nan)

distribution_stats = pd.DataFrame({
    'Feature': numerical_cols,
    'Skewness': skewness_values.values,
    'Kurtosis': kurtosis_values.values
})

print("="*80)
print("Distribution Characteristics (Skewness & Kurtosis)")
print("="*80)
print(distribution_stats.to_string(index=False))



## Visualizing Numerical Feature Distributions

To better understand the data, we plot the **distributions of all numerical features**:

- This helps to **see skewness, outliers, and patterns** visually  
- KDE and histogram plots can reveal whether **transformations are needed**  


In [ ]:
#Plot distributions

number_features = len(numerical_cols)
number_cols = 3
number_rows = (number_features + number_cols -1)// number_cols

fig, axes = plt.subplots(number_rows, number_cols, figsize=(18, number_rows * 4))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    if col in df.columns:
        axes[idx].hist(df[col], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
        axes[idx].axvline(df[col].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df[col].mean():.2f}')
        axes[idx].axvline(df[col].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df[col].median():.2f}')
        axes[idx].set_xlabel(col.replace('_', ' ').title(), fontsize=11)
        axes[idx].set_ylabel('Frequency', fontsize=11)
        axes[idx].set_title(f'Distribution of {col.replace("_", " ").title()}', fontsize=12, fontweight='bold')
        axes[idx].legend()
        axes[idx].grid(alpha=0.3)
for idx in range(len(numerical_cols), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()



In [ ]:
#KDE plots
fig, axes = plt.subplots(5, 2, figsize=(16, 20))
axes = axes.flatten()

selected_features = ['energy', 'tempo', 'danceability', 'loudness', 'liveness', 'valence','duration_ms','acousticness', 'track_popularity', 'speechiness' ]

for idx, col in enumerate(selected_features):
    if col in df.columns:
        sns.kdeplot(data=df, x=col, fill=True, ax=axes[idx], color='steelblue')
        axes[idx].set_title(f'KDE Plot: {col.replace("_", " ").title()}', fontsize=13, fontweight='bold')
        axes[idx].set_xlabel(col.replace('_', ' ').title(), fontsize=11)
        axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Transforming `duration_ms`

From the histograms and KDE plots we saw that `duration_ms` is **strongly right‑skewed**: most songs are relatively short, with a few very long tracks.

To handle this we:

- apply a **log transform** (`duration_log = log1p(duration_ms)`) to compress the long right tail  
- then standardize the result to **`duration_scaled`** using `StandardScaler`, so that duration is on a similar scale as the other numerical features  
- keep **`duration_scaled` as the final engineered duration feature** for any later models  
- drop the original raw `duration_ms` and the intermediate `duration_log` columns to avoid **redundancy and multicollinearity**  


In [ ]:
df['duration_log'] = np.log1p(df['duration_ms'])

scaler = StandardScaler()
df['duration_scaled'] = scaler.fit_transform(df[['duration_log']])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original
sns.histplot(df['duration_ms'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title("Original duration_ms")
axes[0].grid(alpha=0.3)

# After log-transform
sns.histplot(df['duration_log'], kde=True, ax=axes[1], color='steelblue')
axes[1].set_title("After Log Transform")
axes[1].grid(alpha=0.3)

# After scaling
sns.histplot(df['duration_scaled'], kde=True, ax=axes[2], color='navy')
axes[2].set_title("After Log + Standard Scaling")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()



In [ ]:
df_non_numeric = df.select_dtypes(exclude=['number'])
df_non_numeric.columns
print(df.columns)
df.drop(columns=['duration_ms', 'duration_log'], inplace=True)


## Applying Transformations to Skewed Features

Using the skewness and kurtosis table above, we noticed that several audio features are **heavily skewed**:

- `acousticness`, `liveness`, `speechiness`, `instrumentalness` are extremely **right‑skewed** (many values near 0 with a long right tail)  
- `loudness` is **left‑skewed**

To make these variables easier for many ML models to work with, we:

- apply a **log transform** to `acousticness`, `liveness`, and `speechiness` (they are in \[0,1\] and concentrated near 0)  
- apply a **Yeo–Johnson power transform** to `instrumentalness` and `loudness` (they include zeros/negatives, so plain log is not appropriate)  
- create new columns: `acousticness_log`, `liveness_log`, `speechiness_log`, `instrumentalness_yj`, `loudness_yj`  
- drop the original versions of these five columns, so that the transformed versions become our **main engineered features**

Other numerical features are either reasonably symmetric or not as extremely skewed, so we leave them as they are.  


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import PowerTransformer

cols = ["loudness", "liveness", "speechiness", "instrumentalness", "acousticness"]

fig, axes = plt.subplots(len(cols), 2, figsize=(14, 4 * len(cols)))

# BEFORE transformation
for i, col in enumerate(cols):
    sns.kdeplot(df[col], ax=axes[i,0], fill=True, color='skyblue')
    s_before = skew(df[col])
    k_before = kurtosis(df[col])
    axes[i,0].set_title(f"{col} - Before\nskew={s_before:.2f}, kurt={k_before:.2f}")

# TRANSFORMATIONS

# log1p transformations for moderate skew
log_transform_cols = ["acousticness", "liveness", "speechiness"]
for col in log_transform_cols:
    df[col + "_log"] = np.log1p(df[col])

# Yeo-Johnson for loudness and instrumentalness
pt = PowerTransformer(method="yeo-johnson")
df["loudness_yj"] = pt.fit_transform(df[["loudness"]])
pt2 = PowerTransformer(method="yeo-johnson")
df["instrumentalness_yj"] = pt2.fit_transform(df[["instrumentalness"]])

# AFTER transformation
cols_after = [ "loudness_yj", "liveness_log", "speechiness_log", "instrumentalness_yj", "acousticness_log"]

for i, col_after in enumerate(cols_after):
    sns.kdeplot(df[col_after], ax=axes[i,1], fill=True, color='lightgreen')
    s_after = skew(df[col_after])
    k_after = kurtosis(df[col_after])
    axes[i,1].set_title(f"{col_after} - After\nskew={s_after:.2f}, kurt={k_after:.2f}")

plt.tight_layout()
plt.show()

# DROP old columns
df = df.drop(columns=cols)


## Pairplot of `duration_scaled` vs Other Numerical Features

To explore potential relationships, we create a **pairplot** of the `duration_scaled` column against other numerical features:

- Helps to **visualize correlations** between `duration_scaled` and other features  
- Reveals **linear or non-linear relationships**  


In [ ]:
cols_to_exclude = ['duration_scaled', 'duration_ms', 'duration_log']

cols = [col for col in df.select_dtypes(include='number').columns 
        if col not in cols_to_exclude]

n = len(cols)
cols_per_row = 3
rows = (n + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(rows, cols_per_row, figsize=(18, rows * 4))
axes = axes.flatten()

for i, col in enumerate(cols):
    sns.scatterplot(data=df, x='duration_scaled', y=col, ax=axes[i])
    axes[i].set_title(f'duration_scaled vs {col}')
    axes[i].grid(alpha=0.3)

# Turn off empty plots
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


## Observing Correlations

After plotting the pairplot of `duration_scaled` against other numerical features:

- Unfortunately, we **do not observe any strong correlations** with other features  
- This indicates that `duration_scaled` is **independent** of the other numerical features  
- It may still be useful in modeling, but likely **does not explain variance** in other features


## Analyzing Categorical Feature `playlist_genre` with Boxplots

Next, we focus on the **categorical feature `playlist_genre`** and examine its relationship with numerical features:

- Create **boxplots** of each numerical feature grouped by `playlist_genre`  
- Helps to identify **which features tend to be higher or lower** for certain genres  
- Useful for understanding **genre-specific patterns** in the dataset


In [ ]:
cols = ['danceability', 'tempo', 'energy', 'loudness_yj', 'track_popularity']

rows = len(cols)
fig, axes = plt.subplots(rows, 1, figsize=(14, rows * 4)) 

for i, col in enumerate(cols):
    sns.boxplot(data=df, x='playlist_genre', y=col, ax=axes[i])
    axes[i].set_title(f'{col.title()} by Playlist Genre', fontsize=14, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Distribution of playlist_genre by Popularity Group
We visualize the distribution of playlist_genre to identify which genres tend to have the most popular tracks

In [ ]:
plt.figure(figsize=(12,5))
sns.countplot(df,x='playlist_genre', hue='popularity')
plt.title(f'Count of Playlist Genre by Popularity Group')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## Heatmap of Feature Correlations

To get an overall view of the relationships between numerical features, we create a **correlation heatmap**:

- Shows **pairwise correlations** between all numerical features  
- Highlights **strong positive or negative correlations** visually  


In [ ]:
#List of numeric columns to compare with track_popularity
numeric_cols = df.select_dtypes(include='number').columns.tolist()

corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()



## Pairplot of `loudness_yj` vs Other Features

From the correlation heatmap, we observe that `loudness_yj` shows **strong correlations** with many features.  

- To explore this further, we create a **pairplot** of `loudness_yj` against other numerical features  
- Helps to **visualize the relationships** and check for linear or non-linear patterns  
- Useful for understanding how **loudness interacts with other features** in the dataset


In [ ]:
cols_to_compare = [col for col in numeric_cols if col != 'loudness_yj']

# Prepare subplot grid
n = len(cols_to_compare)
cols_per_row = 3
rows = (n + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(rows, cols_per_row, figsize=(18, rows * 4))
axes = axes.flatten()

# Plot loudness vs each numeric feature
for i, col in enumerate(cols_to_compare):
    sns.scatterplot(data=df, x=col, y='loudness_yj', ax=axes[i], alpha=0.6)
    axes[i].set_title(f'Loudness vs {col}')
    axes[i].grid(alpha=0.3)

# Turn off empty subplots if any
for j in range(i+1, len(axes)):
    axes[j].axis('off')
    
plt.tight_layout()
plt.show()

## Observations
- We can observe several linear relationships between **loudness** and other audio features. In general, as loudness increases, many characteristics such as energy and valence tend to increase as well, showing that louder songs often carry stronger musical dynamics.


## Pairplot of liveness vs Other Features

Since liveness is closely related to the popularity of songs, we create a pairplot comparing liveliness with other numerical features to explore potential relationships.

In [ ]:
cols_to_compare = [col for col in numeric_cols if col != 'liveness_log']

# Prepare subplot grid
n = len(cols_to_compare)
cols_per_row = 3
rows = (n + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(rows, cols_per_row, figsize=(18, rows * 4))
axes = axes.flatten()

# Plot liveness_log vs each numeric feature
for i, col in enumerate(cols_to_compare):
    sns.scatterplot(data=df, x=col, y='liveness_log', ax=axes[i], alpha=0.6)
    axes[i].set_title(f'Liveness vs {col}')
    axes[i].grid(alpha=0.3)

# Turn off empty subplots if any
for j in range(i+1, len(axes)):
    axes[j].axis('off')
    
plt.tight_layout()
plt.show()



## Observations
- Unfortunately, we could not identify any meaningful correlations between **liveness** and the other features in the dataset.


## Pairplot of track_popularity vs Other Features

In [ ]:
cols_to_compare = [col for col in numeric_cols if col != 'track_popularity']

# Prepare subplot grid
n = len(cols_to_compare)
cols_per_row = 3
rows = (n + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(rows, cols_per_row, figsize=(18, rows * 4))
axes = axes.flatten()

# Plot track popularity vs each numeric feature
for i, col in enumerate(cols_to_compare):
    sns.scatterplot(data=df, x=col, y='track_popularity', ax=axes[i], alpha=0.6)
    axes[i].set_title(f'track_popularity vs {col}')
    axes[i].grid(alpha=0.3)

# Turn off empty subplots if any
for j in range(i+1, len(axes)):
    axes[j].axis('off')
    
plt.tight_layout()
plt.show()


## Observations
- Track popularity also did not show any meaningful correlation with the other features, indicating that popularity is influenced by external factors not captured in our dataset.


## Transforming track_popularity

The track_popularity column contains relatively large values compared to the other features, and its distribution is also right-skewed:
So we need to use log transformation first
Then standardize the values using StandardScaler

In [ ]:
# Log-transform
df['track_popularity_log'] = np.log1p(df['track_popularity'])

# Scale
df['track_popularity_scaled'] = scaler.fit_transform(df[['track_popularity_log']])

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original
sns.histplot(df['track_popularity'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title("Original track_popularity")
axes[0].grid(alpha=0.3)

# After log-transform
sns.histplot(df['track_popularity_log'], kde=True, ax=axes[1], color='steelblue')
axes[1].set_title("After Log Transform")
axes[1].grid(alpha=0.3)

# After scaling
sns.histplot(df['track_popularity_scaled'], kde=True, ax=axes[2], color='navy')
axes[2].set_title("After Log + Standard Scaling")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Drop original + log column
df.drop(columns=['track_popularity', 'track_popularity_log'], inplace=True)


## Transforming tempo

The tempo column also contains large values compared to the other features, but it has relatiely normal distribution
So we can only apply StandardScaler

In [ ]:
# Scale
df['tempo_scaled'] = scaler.fit_transform(df[['tempo']])

# Plot
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Original
sns.histplot(df['tempo'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title("Original tempo")
axes[0].grid(alpha=0.3)

# After scaling
sns.histplot(df['tempo_scaled'], kde=True, ax=axes[1], color='navy')
axes[1].set_title("After Standard Scaling")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Drop original column
df.drop(columns=['tempo'], inplace=True)


## EDA Summary and Link to Modeling

Summarizing the EDA decisions so far:

- We checked **summary statistics, distributions, skewness and kurtosis** for all numerical features.  
- Strongly skewed variables were **log/Yeo–Johnson transformed**, and the transformed versions replaced the raw ones.  
- For `duration_ms` we engineered a standardized feature `duration_scaled` and removed the raw and intermediate
  versions to keep the dataset compact and non‑redundant.  
- We engineered a few higher‑level features such as `mood_score`, `is_high_energy`, and `is_acoustic_track` that
  capture more **interpretable musical concepts** (energy, mood, acoustic character).
- We identified the features that are most strongly correlated with song popularity.


## K-Means Clustering on Spotify Songs

In this section, we apply K-Means clustering to group songs based on two audio features: **Energy** and **Loudness**. 
These features are chosen because they are strongly related and they represent the “feel” of a song. 

We will:

1. Use the **Elbow Method** to determine the optimal number of clusters.
2. Apply K-Means and visualize the results.
3. Analyze each cluster to understand its characteristics.


### Feature Selection and Scaling

K-Means works better when features are on the same scale. 
As we already scaled the `loudness`, we standardize only `energy`. 


In [ ]:
# Scale 
scaler = StandardScaler()
df['energy_scaled'] = scaler.fit_transform(df[['energy']])

X_scaled = df[['energy_scaled', 'loudness_yj']]
X_scaled_array = X_scaled.to_numpy() 

### Determine Optimal Number of Clusters

We use the **Elbow Method**. 
The "elbow" point in the plot suggests the optimal k.


In [ ]:
# Calculate WCSS for k = 1 to 10
wcss = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Plot the Elbow Method
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('WCSS')
plt.grid(True)
plt.show()


### Apply K-Means Clustering

From the elbow plot, we choose **k = 3**. 
We fit K-Means and assign cluster labels to each song.


In [ ]:
# Choose k based on the elbow
k = 3
kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42)
y_kmeans = kmeans.fit_predict(X_scaled)

# Add cluster labels to the dataframe
df['Cluster'] = y_kmeans


### Visualize Clusters

We visualize the clusters on a 2D scatter plot with centroids marked in red.


In [ ]:
plt.figure(figsize=(8, 6))

# Use proper indexing
sns.scatterplot(
    x=X_scaled_array[:, 0], 
    y=X_scaled_array[:, 1],
    hue=y_kmeans,
    palette='viridis',
    s=100
)

# Plot centroids
centroids = kmeans.cluster_centers_
plt.scatter(
    centroids[:, 0],
    centroids[:, 1],
    c='red',
    s=200,
    marker='X',
    label='Centroids'
)

plt.title('K-Means Clustering of Spotify Songs')
plt.xlabel('Energy (Standardized)')
plt.ylabel('Loudness (Standardized)')
plt.legend()
plt.grid(True)
plt.show()


### Cluster Analysis

We calculate the mean values of `energy` and `loudness` in each cluster to understand their characteristics.

In [ ]:
df.groupby('Cluster')[['energy_scaled', 'loudness_yj']].mean()


## Cluster Interpretation

- **Cluster 0:** Low energy, low loudness → calmer, quieter songs.  
- **Cluster 1:** High energy, high loudness → loud, energetic, intense songs suitable for parties or workouts.  
- **Cluster 2:** Slightly below average energy and loudness → moderate, mid-tempo songs.

This shows that clustering successfully grouped songs by their intensity and volume, helping us identify different types of music based on audio features.


### Creating a pairplot to quickly discover correlations to analyse in depth

In [ ]:
categorical_columns = ['mode','key','playlist_subgenre','playlist_genre','popularity']

numerical_columns = ['track_popularity_scaled','acousticness_log','duration_scaled','instrumentalness_yj','speechiness_log','valence','liveness_log','loudness_yj','danceability','tempo_scaled',
                   'energy_scaled']

pairGrid = sns.PairGrid(df, vars=numerical_columns, hue='popularity', palette='Set1')
pairGrid.map_lower(sns.scatterplot, alpha=0.5, s=20)
pairGrid.map_diag(sns.kdeplot)

# Hide / remove upper triangle
for i in range(len(numerical_columns)):
    for j in range(i+1, len(numerical_columns)):
        pairGrid.axes[i, j].set_visible(False)

pairGrid.add_legend()
plt.show()

## By analyzing the pairplots, we can observe the following relationships:
- We can see a clear correlation between loudness and energy. Songs with higher energy and louder sound tend to have a greater chance of becoming popular.
- In addition, we can say that the louder and more danceable a song is, the greater its chance of becoming highly popular.
- Highly popular songs tend to have lower levels of instrumentalness and acousticness

### Plotting the distribution of songs by features that seem most correlated with track popularity, grouped by popularity


In [ ]:
ft = ['danceability','energy_scaled','loudness_yj', 'instrumentalness_yj', 'acousticness_log']

# Set up the matplotlib figure
fig, axes = plt.subplots(len(ft), 1, figsize=(10, len(ft)*3))

# Plot histograms
for idx, feature in enumerate(ft):
    sns.histplot(data=df, x=feature, hue='popularity', kde=True, ax=axes[idx])
    axes[idx].set_title(f'Distribution of {feature} by Popularity Label')

plt.tight_layout()
plt.show()

### Conclusion
The most correlated features with track popularity are:
- Energy
- Danceability
- Loudness

And also:
- Instrumentalness
- Acousticness

which are negatively correlated.

### Logistic Regression
We will try to apply Logistic Regression model using the most correlated features to predict popularity of songs.

As we observed in the EDA, the features most correlated with popularity are:
- Energy
- Danceability
- Loudness
- Instrumentalness
- Acousticness

In [ ]:
#Select features
FEATURES = [
    'energy_scaled',
    'danceability',
    'loudness_yj',
    'instrumentalness_yj',
    'acousticness_log'
]

X = df[FEATURES]
y = df['popularity']

#Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

#Logistic Regression
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)

print("\n=== Logistic Regression Results ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr, pos_label="High"))
print("Recall:", recall_score(y_test, y_pred_lr, pos_label="High"))
print("F1:", f1_score(y_test, y_pred_lr, pos_label="High"))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))

### Random Forest
Now, let's apply a Random Forest model to classify song popularity

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    min_samples_split=2
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest Results ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf, pos_label="High"))
print("Recall:", recall_score(y_test, y_pred_rf, pos_label="High"))
print("F1:", f1_score(y_test, y_pred_rf, pos_label="High"))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

#Feature Importance (Random Forest)
print("\n=== Feature Importances (RF) ===")
for feat, imp in zip(FEATURES, rf.feature_importances_):
    print(f"{feat:20} : {imp:.4f}")

### Conclusion
By comparing the results of the two models, we can conclude that the Random Forest model performs better than the Logistic Regression model and makes more accurate predictions.